# ⚡ DataGrid Energy Predictor — Walkthrough

In this notebook I walk through the full pipeline I built to analyze the Alberta electrical grid and identify which energy assets have the most room for improvement.

Here's what I cover:

1. **Data Loading** — loading the asset data I extracted from the Alberta CSD report
2. **Exploratory Data Analysis** — getting a feel for the data before modeling
3. **Feature Engineering** — creating the efficiency and optimization targets
4. **Feature Selection** — figuring out which features actually matter
5. **Model Training** — training and comparing 4 models (LR, RF, XGBoost, PyTorch NN)
6. **Recommendations** — ranking assets by how much they could improve

> Make sure you've run `pip install -r requirements.txt` before starting.

## 0. Imports & Setup

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from xgboost import XGBRegressor

# Set consistent style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

# Paths
BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, 'data')

print('✅ Imports successful')
print(f'📁 Data directory: {DATA_DIR}')

## 1. Loading the Data

I extracted all of this from Alberta's **Current Supply & Demand (CSD) Report PDF** using `pdf_data_extractor.py`. Here I'm just loading the CSVs that came out of that step.

In [ ]:
# Load all datasets
assets_df     = pd.read_csv(os.path.join(DATA_DIR, 'all_assets.csv'))
summary_df    = pd.read_csv(os.path.join(DATA_DIR, 'summary.csv'))
generation_df = pd.read_csv(os.path.join(DATA_DIR, 'generation_by_group.csv'))
interchange_df = pd.read_csv(os.path.join(DATA_DIR, 'interchange.csv'))

print(f'Assets dataset:         {assets_df.shape[0]} rows × {assets_df.shape[1]} cols')
print(f'Summary dataset:        {summary_df.shape[0]} rows × {summary_df.shape[1]} cols')
print(f'Generation by group:    {generation_df.shape[0]} rows × {generation_df.shape[1]} cols')
print(f'Interchange data:       {interchange_df.shape[0]} rows × {interchange_df.shape[1]} cols')

In [ ]:
assets_df.head(10)

## 2. Exploring the Data

Before jumping into modeling I wanted to understand the distribution of assets, how capacity varies by category, and where the obvious gaps are.

In [ ]:
# Basic statistics
print('=== Dataset Info ===')
assets_df.info()
print('\n=== Descriptive Statistics ===')
assets_df.describe()

In [ ]:
# Asset count by category
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

category_counts = assets_df['category'].value_counts()
axes[0].bar(category_counts.index, category_counts.values, color=sns.color_palette('muted', len(category_counts)))
axes[0].set_title('Number of Assets by Category', fontsize=13)
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Total capacity by category
capacity_by_cat = assets_df.groupby('category')['maximum_capability'].sum().sort_values(ascending=False)
axes[1].bar(capacity_by_cat.index, capacity_by_cat.values, color=sns.color_palette('muted', len(capacity_by_cat)))
axes[1].set_title('Total Capacity by Category (MW)', fontsize=13)
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Total Maximum Capability (MW)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Distribution of Maximum Capability vs Total Net Generation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(assets_df['maximum_capability'], bins=20, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title('Distribution of Maximum Capability (MW)')
axes[0].set_xlabel('MW')
axes[0].set_ylabel('Frequency')

axes[1].hist(assets_df['total_net_generation'], bins=20, color='seagreen', edgecolor='white', alpha=0.85)
axes[1].set_title('Distribution of Total Net Generation (MW)')
axes[1].set_xlabel('MW')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Scatter: Capability vs Generation (coloured by category)
plt.figure(figsize=(10, 7))
categories = assets_df['category'].unique()
palette = sns.color_palette('tab10', len(categories))

for i, cat in enumerate(categories):
    subset = assets_df[assets_df['category'] == cat]
    plt.scatter(subset['maximum_capability'], subset['total_net_generation'],
                label=cat, color=palette[i], alpha=0.75, s=60)

# Perfect efficiency line
max_val = assets_df['maximum_capability'].max()
plt.plot([0, max_val], [0, max_val], 'k--', alpha=0.4, label='100% efficiency')

plt.xlabel('Maximum Capability (MW)', fontsize=12)
plt.ylabel('Total Net Generation (MW)', fontsize=12)
plt.title('Asset Capability vs. Actual Generation\n(Points below dashed line = underperforming)', fontsize=13)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 3. Feature Engineering

I created two target variables that capture what I actually care about:
- **Efficiency Ratio** = Total Net Generation ÷ Maximum Capability — how close to full capacity is the asset running? *(0–1, higher is better)*
- **Optimization Potential** = Maximum Capability − Total Net Generation — how many MW is this asset leaving unused?

In [ ]:
df = assets_df.copy()

# Target variables
df['efficiency_ratio'] = np.where(
    df['maximum_capability'] > 0,
    df['total_net_generation'] / df['maximum_capability'],
    0
)
df['optimization_potential'] = df['maximum_capability'] - df['total_net_generation']

print('Target variable statistics:')
df[['efficiency_ratio', 'optimization_potential']].describe().round(3)

In [ ]:
# Efficiency by category
eff_by_cat = df.groupby('category')['efficiency_ratio'].mean().sort_values()

plt.figure(figsize=(10, 5))
bars = plt.barh(eff_by_cat.index, eff_by_cat.values,
                color=['#d62728' if v < 0.6 else '#2ca02c' for v in eff_by_cat.values])
plt.axvline(x=0.8, color='gray', linestyle='--', alpha=0.7, label='80% target')
plt.xlabel('Average Efficiency Ratio', fontsize=12)
plt.title('Average Efficiency Ratio by Energy Category', fontsize=13)
plt.legend()

for bar, val in zip(bars, eff_by_cat.values):
    plt.text(val + 0.01, bar.get_y() + bar.get_height()/2,
             f'{val:.1%}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Build feature matrix
category_dummies = pd.get_dummies(df['category'], prefix='category')
numerical = df[['maximum_capability', 'total_net_generation', 'dispatched_contingency_reserve']]
X = pd.concat([numerical, category_dummies], axis=1)
feature_names = X.columns.tolist()

y_efficiency  = df['efficiency_ratio'].values
y_potential   = df['optimization_potential'].values

print(f'Feature matrix shape: {X.shape}')
print(f'Features: {feature_names}')

## 4. Feature Correlation Analysis

Before selecting features I wanted to see how each variable relates to the targets. Strong correlations help confirm which features are worth keeping.

In [ ]:
# Correlation heatmap
combined = X.copy()
combined['efficiency_ratio'] = y_efficiency
combined['optimization_potential'] = y_potential

corr = combined.corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Top correlations with each target
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, target, label in zip(axes,
                              ['efficiency_ratio', 'optimization_potential'],
                              ['Efficiency Ratio', 'Optimization Potential']):
    target_corr = corr[target].drop(['efficiency_ratio', 'optimization_potential']).abs().sort_values(ascending=False)
    colors = ['#e74c3c' if corr[target][f] < 0 else '#2ecc71' for f in target_corr.index]
    ax.bar(target_corr.index, target_corr.values, color=colors)
    ax.set_title(f'Absolute Correlation with {label}', fontsize=12)
    ax.set_xlabel('Feature')
    ax.set_ylabel('|Correlation|')
    ax.tick_params(axis='x', rotation=75)

plt.tight_layout()
plt.show()

## 5. Training & Comparing Models

I trained 4 models on each target and compared them using RMSE, MAE, and R². I wanted to see whether the extra complexity of XGBoost or the neural network actually pays off over a simple Random Forest.

In [ ]:
# Train/test split + scaling
X_train, X_test, y_train_eff, y_test_eff, y_train_pot, y_test_pot = train_test_split(
    X.values, y_efficiency, y_potential, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Training samples : {X_train_s.shape[0]}')
print(f'Test samples     : {X_test_s.shape[0]}')

In [ ]:
# PyTorch Neural Network
class EnergyNet(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x)

def train_nn(X_tr, y_tr, X_te, y_te, epochs=150):
    n = X_tr.shape[1]
    model = EnergyNet(n)
    opt   = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    crit  = nn.MSELoss()
    loader = DataLoader(
        TensorDataset(torch.tensor(X_tr, dtype=torch.float32),
                      torch.tensor(y_tr, dtype=torch.float32).unsqueeze(1)),
        batch_size=16, shuffle=True
    )
    train_losses, val_losses = [], []
    best_loss, best_state, patience, ctr = float('inf'), None, 20, 0

    X_te_t = torch.tensor(X_te, dtype=torch.float32)
    y_te_t = torch.tensor(y_te, dtype=torch.float32).unsqueeze(1)

    for _ in range(epochs):
        model.train()
        ep_loss = 0
        for xb, yb in loader:
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            opt.step()
            ep_loss += loss.item() * len(xb)
        train_losses.append(ep_loss / len(X_tr))

        model.eval()
        with torch.no_grad():
            val = crit(model(X_te_t), y_te_t).item()
        val_losses.append(val)

        if val < best_loss:
            best_loss = val
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            ctr = 0
        else:
            ctr += 1
            if ctr >= patience:
                break

    model.load_state_dict(best_state)
    return model, train_losses, val_losses

def nn_predict(model, X):
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(X, dtype=torch.float32)).squeeze().numpy()

print('EnergyNet architecture:')
print(EnergyNet(X_train_s.shape[1]))

In [ ]:
def evaluate(y_true, y_pred, name):
    mse = mean_squared_error(y_true, y_pred)
    return {
        'Model': name,
        'RMSE':  round(np.sqrt(mse), 4),
        'MAE':   round(mean_absolute_error(y_true, y_pred), 4),
        'R²':    round(r2_score(y_true, y_pred), 4)
    }

results = {'efficiency_ratio': [], 'optimization_potential': []}
trained_models = {'efficiency_ratio': {}, 'optimization_potential': {}}
nn_curves = {}

for target_name, y_train, y_test in [
    ('efficiency_ratio',    y_train_eff, y_test_eff),
    ('optimization_potential', y_train_pot, y_test_pot)
]:
    print(f'\n--- Training models for: {target_name} ---')

    lr = LinearRegression().fit(X_train_s, y_train)
    rf = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train_s, y_train)
    xgb = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbosity=0).fit(X_train_s, y_train)
    nn_model, train_l, val_l = train_nn(X_train_s, y_train, X_test_s, y_test)

    trained_models[target_name] = {'Linear Regression': lr, 'Random Forest': rf,
                                   'XGBoost': xgb, 'Neural Network': nn_model}
    nn_curves[target_name] = (train_l, val_l)

    for name, model in [('Linear Regression', lr), ('Random Forest', rf), ('XGBoost', xgb)]:
        results[target_name].append(evaluate(y_test, model.predict(X_test_s), name))

    results[target_name].append(evaluate(y_test, nn_predict(nn_model, X_test_s), 'Neural Network'))

    df_res = pd.DataFrame(results[target_name])
    print(df_res.to_string(index=False))

print('\n✅ All models trained.')

In [ ]:
# Learning curves for Neural Network
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, target in zip(axes, ['efficiency_ratio', 'optimization_potential']):
    train_l, val_l = nn_curves[target]
    ax.plot(train_l, label='Train Loss', color='steelblue')
    ax.plot(val_l,   label='Val Loss',   color='tomato')
    ax.set_title(f'NN Learning Curve — {target}', fontsize=12)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Model comparison charts
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for row, target in enumerate(['efficiency_ratio', 'optimization_potential']):
    df_res = pd.DataFrame(results[target])
    model_names = df_res['Model'].tolist()

    axes[row, 0].bar(model_names, df_res['RMSE'], color='steelblue')
    axes[row, 0].set_title(f'RMSE — {target}', fontsize=11)
    axes[row, 0].set_ylabel('Lower is better')
    axes[row, 0].tick_params(axis='x', rotation=15)

    axes[row, 1].bar(model_names, df_res['R²'], color='seagreen')
    axes[row, 1].set_title(f'R² Score — {target}', fontsize=11)
    axes[row, 1].set_ylabel('Higher is better')
    axes[row, 1].tick_params(axis='x', rotation=15)
    axes[row, 1].set_ylim(0, 1.05)

plt.suptitle('Model Performance Comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 6. Feature Importance

I used the Random Forest's built-in feature importances to check which variables the model was actually leaning on. This helps validate the feature selection I did earlier.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, target in zip(axes, ['efficiency_ratio', 'optimization_potential']):
    rf_model = trained_models[target]['Random Forest']
    importances = rf_model.feature_importances_
    feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(10)

    sns.barplot(x=feat_imp.values, y=feat_imp.index, ax=ax, palette='Blues_r')
    ax.set_title(f'RF Feature Importance — {target}', fontsize=12)
    ax.set_xlabel('Importance')

plt.tight_layout()
plt.show()

## 7. Recommendations

I used the best-performing model (Random Forest) to predict the potential efficiency of every asset and ranked them by how much generation could be recovered. This is the actual output I'd hand to a grid operator.

In [ ]:
# Predict efficiency and potential for all assets
X_all_scaled = scaler.transform(X.values)

rf_eff = trained_models['efficiency_ratio']['Random Forest']
rf_pot = trained_models['optimization_potential']['Random Forest']

recs = df.copy()
recs['predicted_efficiency']  = rf_eff.predict(X_all_scaled)
recs['predicted_potential']   = rf_pot.predict(X_all_scaled)
recs['efficiency_improvement'] = np.maximum(0, recs['predicted_efficiency'] - recs['efficiency_ratio'])
recs['potential_generation_increase'] = recs['efficiency_improvement'] * recs['maximum_capability']

top10 = recs.sort_values('potential_generation_increase', ascending=False).head(10)

# Summary stats
total_increase   = recs['potential_generation_increase'].sum()
current_total    = recs['total_net_generation'].sum()
pct_increase     = (total_increase / current_total) * 100 if current_total > 0 else 0

print(f'Current total generation :  {current_total:,.0f} MW')
print(f'Potential generation gain:  {total_increase:,.0f} MW')
print(f'Percentage improvement   :  {pct_increase:.1f}%')

In [ ]:
# Top 10 assets table
top10[['name', 'category', 'maximum_capability', 'total_net_generation',
       'efficiency_ratio', 'predicted_efficiency', 'potential_generation_increase']].round(3)

In [ ]:
# Visualization: Top 10 assets by potential gain
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Bar chart — potential generation increase
axes[0].barh(top10['name'], top10['potential_generation_increase'], color='steelblue')
axes[0].set_xlabel('Potential Generation Increase (MW)', fontsize=11)
axes[0].set_title('Top 10 Assets: Potential Generation Gain', fontsize=12)
axes[0].invert_yaxis()

# Grouped bar chart — current vs potential efficiency
y_pos   = np.arange(len(top10))
bar_w   = 0.35
axes[1].barh(y_pos,          top10['efficiency_ratio'].values,    bar_w, label='Current Efficiency',   color='#e74c3c', alpha=0.85)
axes[1].barh(y_pos + bar_w,  top10['predicted_efficiency'].values, bar_w, label='Predicted Efficiency', color='#2ecc71', alpha=0.85)
axes[1].set_yticks(y_pos + bar_w / 2)
axes[1].set_yticklabels(top10['name'].values, fontsize=9)
axes[1].set_xlabel('Efficiency Ratio', fontsize=11)
axes[1].set_title('Current vs. Predicted Efficiency', fontsize=12)
axes[1].legend()
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Potential gain by category
cat_gain = recs.groupby('category')['potential_generation_increase'].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
plt.bar(cat_gain.index, cat_gain.values,
        color=sns.color_palette('muted', len(cat_gain)))
plt.xlabel('Category', fontsize=12)
plt.ylabel('Potential Generation Increase (MW)', fontsize=12)
plt.title('Optimization Potential by Energy Category', fontsize=13)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 8. What I Learned

| Takeaway | Detail |
|----------|--------|
| Best model | Random Forest — best R², lowest RMSE across both targets |
| Strongest predictor of efficiency | `total_net_generation`, `category_COGENERATION` |
| Strongest predictor of potential | `maximum_capability`, `total_net_generation` |
| Total recoverable generation | See output above |

### What I'd Do Next
- Bring in **time-series data** to track how efficiency changes over time for each asset
- Add **weather features** (temperature, wind speed) — especially relevant for solar and wind assets
- Wrap this into a **REST API** so it can serve real-time recommendations
- Try extending it to other grids like BC Hydro or Ontario IESO